# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

### Deliverable 1 - open the tunnel first

The Kafka broker is not reachable directly; `cs544-f26.cs.uic.edu` is. The
command below opens an SSH tunnel forwarding **local** port 9092 to port 9092
*as seen from the server*, so `localhost:9092` on this machine reaches the
broker:

```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

`-L 9092:localhost:9092` forwards local 9092 -> localhost:9092 on the server;
`-N` no remote command, `-T` no pseudo-terminal, `-f` background, and
`ServerAliveInterval=60` keeps an idle tunnel from being dropped.

Verify the tunnel and list the broker's topics:

```
kcat -b localhost:9092 -L
```

Close it when done:

```
lsof -ti:9092 | xargs kill -9
```


In [1]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-c'   # <-- change the letter if your section is not C

# The SSH tunnel forwards localhost:9092 on this machine to port 9092 on
# cs544-f26.cs.uic.edu, so from Python the broker simply looks local.
bootstrap_server = 'localhost:9092'


## Deliverable 1 - Topics and Offsets

**Topic.** A topic is the named, append-only log Kafka organises messages into -
here `recitation-c`. Producers append to the end of a topic; consumers read
forward through it. A topic is split into one or more *partitions*, and the
partition is the unit that actually preserves order: messages are appended to
the end of a partition and never modified. Reading a message does **not** remove
it - the log is retained for a configured period, which is what makes replay
possible.

**Offset.** Every message in a partition gets a monotonically increasing integer
id - its offset (0, 1, 2, ...). The offset is the permanent address of a message
within its partition, so a consumer's position is just a number: "I have read up
to offset N."

**Why offsets give continuity after a disconnect.** A consumer that sets a
`group_id` periodically *commits* its position back to Kafka (here every
`auto_commit_interval_ms=1000` ms, because `enable_auto_commit=True`). Kafka
stores that committed offset per (consumer group, topic, partition) in the
internal `__consumer_offsets` topic - on the broker, not in the client. So if
the consumer crashes, the SSH tunnel drops, or the kernel is restarted, the
messages produced meanwhile are still sitting in the log. When the consumer
reconnects with the same `group_id`, Kafka hands back its last committed offset
and delivery resumes from there - nothing is lost, and nothing already processed
is re-read.

`auto_offset_reset` only matters when there is *no* committed offset for the
group (a brand-new group, or one whose offsets have expired):

- `earliest` - start at the lowest retained offset and replay the whole log
- `latest` - start at the end, seeing only messages produced from now on

Because commits happen on an interval rather than per message, the guarantee is
*at-least-once*: a crash between processing a message and committing its offset
means that message is delivered again on restart.

**Demo for the TA:** run the consumer, interrupt it, run the producer again,
then re-run the consumer - it picks up only the new messages. Change the
`group_id` suffix and it replays everything from the earliest offset instead.


### Producer Mode -> Writes Data to Broker

In [2]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# The producer connects through the SSH tunnel at localhost:9092.
# value_serializer turns each Python object into the bytes Kafka stores:
# dumps() -> JSON text, .encode('utf-8') -> bytes.
producer = KafkaProducer(bootstrap_servers=[bootstrap_server],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# Cities of my choice
cities = ['Chicago', 'Pittsburgh', 'New York']

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    # send() appends the message to the end of the topic's log and returns
    # immediately; the record gets the next offset in its partition.
    producer.send(topic=topic, value=data)
    sleep(1)

# send() is asynchronous and buffered, so flush() before the cell ends to make
# sure every message actually reached the broker.
producer.flush()
print("Done writing.")


Writing to Kafka Broker
Writing: 2026-09-25 11:38:58,Pittsburgh,28ºC


Writing: 2026-09-25 11:38:59,Pittsburgh,25ºC


Writing: 2026-09-25 11:39:00,Chicago,25ºC


Writing: 2026-09-25 11:39:01,Pittsburgh,31ºC


Writing: 2026-09-25 11:39:02,Pittsburgh,25ºC


...

Writing: 2026-09-25 11:39:04,New York,21ºC


Writing: 2026-09-25 11:39:05,Chicago,32ºC


Writing: 2026-09-25 11:39:06,New York,23ºC


Writing: 2026-09-25 11:39:07,New York,26ºC


Done writing.


### Consumer Mode -> Reads Data from Broker

In [3]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

consumer = KafkaConsumer(
    # 1st positional arg = the topic(s) to subscribe to.
    topic,
    # Same tunnelled broker address the producer used.
    bootstrap_servers=[bootstrap_server],
    # Where to start when this group has NO committed offset yet:
    #   'earliest' -> replay the topic from the very first retained message
    #   'latest'   -> only messages produced from now on
    # Once the group HAS a committed offset this setting is ignored and the
    # consumer resumes from that offset instead - that is what gives message
    # continuity across a disconnect.
    auto_offset_reset='earliest',
    # The consumer group this consumer belongs to. Kafka stores committed
    # offsets per (group, topic, partition), so a group_id is what makes
    # "resume where I left off" possible. Change the suffix to force a
    # full replay from the earliest offset again.
    group_id=f'{topic}-demo-group',
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000,
    # Stop blocking after 10s of no new messages so the cell finishes and we
    # can inspect kafka_log.csv. Remove this to stream forever (Ctrl-C to stop).
    consumer_timeout_ms=10000,
)

print('Reading Kafka Broker')
for record in consumer:
    # record carries the metadata; record.value carries the payload.
    # Default record.value type is bytes!
    #   .decode()  -> the JSON *text* the producer wrote, e.g. '"...,25\\u00baC"'
    #   loads(...)  -> back to the original Python string, '...,25\u00baC'
    message = record.value.decode()
    value = loads(message)
    print(f'[partition {record.partition} offset {record.offset}]', value)

    # Append the consumed message to the CSV log.
    #
    # NOTE: the starter line was
    #     os.system(f"echo {message} >> kafka_log.csv")
    # which writes `message` - the raw JSON text - not the decoded value. Since
    # json.dumps() defaults to ensure_ascii=True, the degree sign is escaped, so
    # that line lands the literal 6 characters \u00ba in the file instead of
    # the real º. The print() above looks correct, which hides the bug. Writing
    # `value` from Python keeps the real character and also avoids handing an
    # unquoted string to the shell.
    with open('kafka_log.csv', 'a', encoding='utf-8') as f:
        f.write(value + '\n')

consumer.close()
print('Stopped reading.')


Reading Kafka Broker


[partition 0 offset 20] 2026-09-25 11:38:58,Pittsburgh,28ºC
[partition 0 offset 21] 2026-09-25 11:38:59,Pittsburgh,25ºC
[partition 0 offset 22] 2026-09-25 11:39:00,Chicago,25ºC
[partition 0 offset 23] 2026-09-25 11:39:01,Pittsburgh,31ºC
[partition 0 offset 24] 2026-09-25 11:39:02,Pittsburgh,25ºC
[partition 0 offset 25] 2026-09-25 11:39:03,New York,32ºC
[partition 0 offset 26] 2026-09-25 11:39:04,New York,21ºC
[partition 0 offset 27] 2026-09-25 11:39:05,Chicago,32ºC
[partition 0 offset 28] 2026-09-25 11:39:06,New York,23ºC
[partition 0 offset 29] 2026-09-25 11:39:07,New York,26ºC


Stopped reading.


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [4]:
# kcat command: connect to the local Kafka broker, specify a topic,
# and consume messages from the earliest offset.
#
#   kcat -b localhost:9092 -t recitation-c -C -o beginning -e
#
#   -b localhost:9092  broker to connect to (our SSH-tunnelled local port)
#   -t recitation-c    the topic to read
#   -C                 consumer mode (-P would be producer mode)
#   -o beginning       start at the EARLIEST offset in the partition, i.e.
#                      replay the whole retained log instead of only new messages
#   -e                 exit once the last message is reached (drop it to tail
#                      the topic and keep waiting for new messages)
#
# Note: kcat spells the earliest offset "beginning". "earliest" is the Kafka
# client-config wording used by auto_offset_reset above - a different
# vocabulary. Careful: `-o earliest` does NOT error; kcat has no such keyword,
# so it falls through to the numeric parse and means absolute offset 0. That
# looks the same on a fresh topic, but once retention trims the head of the log
# the lowest valid offset is > 0 and offset 0 is out of range. Use "beginning".
#
# Run it in a terminal, or straight from this notebook:
!kcat -b localhost:9092 -t {topic} -C -o beginning -e


"2026-09-25 11:38:58,Pittsburgh,28\u00baC"
"2026-09-25 11:38:59,Pittsburgh,25\u00baC"
"2026-09-25 11:39:00,Chicago,25\u00baC"
"2026-09-25 11:39:01,Pittsburgh,31\u00baC"
"2026-09-25 11:39:02,Pittsburgh,25\u00baC"
"2026-09-25 11:39:03,New York,32\u00baC"
"2026-09-25 11:39:04,New York,21\u00baC"
"2026-09-25 11:39:05,Chicago,32\u00baC"
"2026-09-25 11:39:06,New York,23\u00baC"
"2026-09-25 11:39:07,New York,26\u00baC"


% Reached end of topic recitation-c [0] at offset 30: exiting
